In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob as glob
from pathlib import Path
import re
import os
import sys
from collections import defaultdict
from collections import defaultdict
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator, LogLocator, FuncFormatter)
from scipy import interpolate
import matplotlib as mpl
import matplotlib.lines as lines
# from skbio.diversity.alpha import shannon, simpson, chao1, fisher_alpha, hill


In [ ]:
project_dir = '/tcr_seq' # change this to your project directory
output_dir = os.path.join(project_dir, "outputs", 'boxplot_cancer_types')

In [5]:
tcr_df_dict = defaultdict(pd.DataFrame)

for path in glob.glob(os.path.join(project_dir, '**', '*_unique_combined_meta_df.csv'), recursive=True):
    name = os.path.basename(path).split('_unique_combined_meta_df.csv')[0]
    tcr_df_dict[name] = pd.read_csv(path)

/var/folders/ym/3qfzwxc96s3c9_tk6z79_r1m0000gn/T/ipykernel_53894/452630753.py:5: DtypeWarning: Columns (49) have mixed types. Specify dtype option on import or set low_memory=False.
  tcr_df_dict[name] = pd.read_csv(path)


In [6]:
len(tcr_df_dict.keys())

20

In [ ]:
for key, df in tcr_df_dict.items():
    print(key, "tumor_type" in df.columns)

In [ ]:
tcr_df_dict['NSCLC'].loc[:, 'CD4'] = tcr_df_dict['NSCLC'].loc[:, 'CD8_or_CD4'] == 'CD4'
tcr_df_dict['NSCLC'].loc[:, 'CD8'] = tcr_df_dict['NSCLC'].loc[:, 'CD8_or_CD4'] == 'CD8'

In [9]:
import pandas as pd

rows = []

for key, df in tcr_df_dict.items():
    tmp = (
        df[df["CD8"] == True]
        .groupby("patient")
        .agg(
            n_cd8=("CD8", "size"),
            tumor_type=("tumor_type", "first"),
            # q_1=("unique_clonotype_count_aa", lambda x: hill(x, order=1)),
            # q_2=("unique_clonotype_count_aa", lambda x: hill(x, order=2)),
            # chao1=("unique_clonotype_count_aa", lambda x: chao1(x, bias_corrected=True)),
            more_or_equal_to_2 =("unique_clonotype_count_aa", lambda x: (x >= 2).sum()),
            total_cd8_cells = ("unique_clonotype_count_aa", "sum"),
            # fraction_expanded_total_cells = ("unique_clonotype_count_aa", lambda x: (x >= 2).sum() / len(x)

        )
        .reset_index()
    )
    tmp["data_source"] = key
    tmp["CD8"] = True  # since we filtered on it
    rows.append(tmp)

cd8_df = pd.concat(rows, ignore_index=True)
# cd8_df.to_csv(os.path.join(output_dir, "cd8_patient_summary.csv"), index=False)
cd8_df

,patient,n_cd8,tumor_type,more_or_equal_to_2,total_cd8_cells,data_source,CD8
0,ICC1,241,prostate,75,737,PRJNA768795,True
1,ICC2,234,prostate,77,649,PRJNA768795,True
2,ICC3,331,prostate,105,923,PRJNA768795,True
3,ICC4,233,prostate,67,835,PRJNA768795,True
4,ICC5,417,prostate,100,1130,PRJNA768795,True
...,...,...,...,...,...,...,...
245,vdjNKI142,171,TNBC,1,172,bellini,True
246,vdjNKI146,511,TNBC,81,915,bellini,True
247,vdjNKI147,700,TNBC,117,1347,bellini,True
248,vdjNKI148,539,TNBC,87,836,bellini,True


In [10]:
rename_dict = {
    "TNBC": "Breast",
    "BC": "Breast",
    "CRC": "Colon",
    "MEL": "Melanoma",
    "OV": "Ovary",
    "RCC": "Kidney clear cell",
    "pancreatic": "Pancreas",
    "PACA": "Pancreas",
    "BLA": "Bladder",
    "ESSC": "Esophagus",
    "THCA": "Thyroid",
    "UCEC": "Uterine",
    "ESCA": "Esophagus",
    "RC": "Kidney clear cell",
    "Lung": "NSCLC",
    "Endo": "Endometrium",
    "FTC": "Thyroid",
    'Renal': "Kidney clear cell",
    'breast': "Breast",
    'prostate': "Prostate"
    }
cd8_df["tumor_type"] = cd8_df["tumor_type"].replace(rename_dict)
cd8_df

,patient,n_cd8,tumor_type,more_or_equal_to_2,total_cd8_cells,data_source,CD8
0,ICC1,241,Prostate,75,737,PRJNA768795,True
1,ICC2,234,Prostate,77,649,PRJNA768795,True
2,ICC3,331,Prostate,105,923,PRJNA768795,True
3,ICC4,233,Prostate,67,835,PRJNA768795,True
4,ICC5,417,Prostate,100,1130,PRJNA768795,True
...,...,...,...,...,...,...,...
245,vdjNKI142,171,Breast,1,172,bellini,True
246,vdjNKI146,511,Breast,81,915,bellini,True
247,vdjNKI147,700,Breast,117,1347,bellini,True
248,vdjNKI148,539,Breast,87,836,bellini,True


In [11]:
filtered_cd8_df = cd8_df[~(cd8_df.loc[:, 'n_cd8'] == 2967)]

In [ ]:
tumor_name_color_df = pd.read_csv('/PanCanAtlasTumors_color_coded_by_organ_system_20170302.tsv', sep='\t')

tumor_name_color_df.ton_name.dropna()


unique_tumor_types = pd.Series(filtered_cd8_df.tumor_type.unique())
unique_tumor_types = unique_tumor_types[unique_tumor_types.isin(tumor_name_color_df.ton_name)]

tumor_name_color_df.loc[tumor_name_color_df.ton_name.isin(unique_tumor_types), :].loc[:, ['ton_name', 'Hex Colors']]

tumor_color_dict = dict(tumor_name_color_df.loc[tumor_name_color_df.ton_name.isin(unique_tumor_types), :].loc[:, ['ton_name', 'Hex Colors']].values.tolist()
)

tumor_color_dict.update({'Thyroid': '#F9ED32', 'NSCLC': '#A084BD', 'Prostate': '#7E1918'})
tumor_color_dict.update({'Lung adenocarcinoma': '#D3C3E0', 'Lung squamous': '#A084BD'})

tumor_color_dict

{'Breast': '#ED2891',
 'Colon': '#9EDDF9',
 'Esophagus': '#007EB5',
 'Pancreas': '#6E7BA2',
 'Endometrium': '#F6B667',
 'Ovary': '#D97D25',
 'Uterine': '#FBE3C7',
 'Melanoma': '#BBD642',
 'Bladder': '#FAD2D9',
 'Kidney clear cell': '#F8AFB3',
 'Thyroid': '#F9ED32',
 'NSCLC': '#A084BD',
 'Prostate': '#7E1918',
 'Lung adenocarcinoma': '#D3C3E0',
 'Lung squamous': '#A084BD'}

In [ ]:
bellini_df = pd.read_csv('/tcr_seq_data/NKI_datasets/bellini/outputs/bellini_unique_combined_meta_df.csv')
patient_list = list(bellini_df[~bellini_df.loc[:, 'Cohort_a'].isin(['C', 'D'])].patient.unique())

# remove non-relevant patients from bellini dataset
# now randomly downsample the marleen kok data

filtered_cd8_df.data_source.value_counts()

# biggest individual data  source is 18 

tmp_df = filtered_cd8_df[filtered_cd8_df.loc[:, 'data_source'] == 'bellini']
print(tmp_df.shape)

tmp_df = tmp_df[tmp_df.patient.isin(patient_list)]

print(tmp_df.shape)

df_downsample  = tmp_df.apply(lambda df: df.sample(n=18, random_state=1))

print(df_downsample.shape)
print(filtered_cd8_df.shape)
print(df_downsample.shape)
print(filtered_cd8_df[filtered_cd8_df.loc[:, 'data_source'] == 'bellini'].shape)
print(filtered_cd8_df[~(filtered_cd8_df.loc[:, 'data_source'] == 'bellini')].shape)

downsampled_filtered_cd8_df = pd.concat([df_downsample, filtered_cd8_df[~(filtered_cd8_df.loc[:, 'data_source'] == 'bellini')]])
print(downsampled_filtered_cd8_df.shape)

(70, 7)
(32, 7)
(18, 7)
(249, 7)
(18, 7)
(70, 7)
(179, 7)
(197, 7)


In [ ]:
tmp_rename_dict = {
    'Uterine': 'Endometrium',
}

downsampled_filtered_cd8_df.loc[:, 'tumor_type'] = (
    downsampled_filtered_cd8_df['tumor_type'].replace(tmp_rename_dict)
)
downsampled_filtered_cd8_df

In [ ]:
downsampled_filtered_cd8_df.loc[
    downsampled_filtered_cd8_df['patient'] == "OV",
    'tumor_type'] = 'Ovary'

downsampled_filtered_cd8_df.reset_index(drop=True, inplace=True)
downsampled_filtered_cd8_df.tumor_type.value_counts()

tumor_type
Breast               43
NSCLC                30
Kidney clear cell    20
Colon                18
Esophagus            14
Pancreas             13
Endometrium          12
Melanoma             11
Ovary                10
Thyroid              10
Bladder               9
Prostate              7
Name: count, dtype: int64

In [ ]:
adeno_list = ['Lung24', 'Lung26', 'Lung18', 'Lung15', 'Lung10', 'Lung11', 'Lung14', 'Lung16', 'Lung12', 'Lung17', 'Lung19', 'Lung1', 'Lung4', 'Lung5', 'Lung6','Lung7', 'Lung8']

squamous_list = ['Lung21', 'Lung22', 'Lung23', 'Lung25', 'Lung2', 'Lung3', 'Lung9']

mask = downsampled_filtered_cd8_df.patient.isin(adeno_list)

downsampled_filtered_cd8_df.loc[mask, "tumor_type"] = "Lung adenocarcinoma"

mask = downsampled_filtered_cd8_df.patient.isin(squamous_list)

downsampled_filtered_cd8_df.loc[mask, "tumor_type"] = "Lung squamous"

downsampled_filtered_cd8_df = downsampled_filtered_cd8_df[~downsampled_filtered_cd8_df.patient.isin(["Lung13", "Lung20", "Lung6"])]

downsampled_filtered_cd8_df.tumor_type.value_counts()
downsampled_filtered_cd8_df.to_csv(os.path.join(output_dir, "cd8_patient_summary_downsampled.csv"), index=False)

### generate df for CD4

In [18]:
### now generate df for CD4

rows = []

for key, df in tcr_df_dict.items():
    tmp = (
        df[df["CD4"] == True]
        .groupby("patient")
        .agg(
            n_cd4=("CD4", "size"),
            tumor_type=("tumor_type", "first"),
            more_or_equal_to_2 =("unique_clonotype_count_aa", lambda x: (x >= 2).sum()),
            total_cd4_cells = ("unique_clonotype_count_aa", "sum")

        )
        .reset_index()
    )
    tmp["data_source"] = key
    tmp["CD4"] = True  # since we filtered on it
    rows.append(tmp)

cd4_df = pd.concat(rows, ignore_index=True)
cd4_df

,patient,n_cd4,tumor_type,more_or_equal_to_2,total_cd4_cells,data_source,CD4
0,ICC1,424,prostate,47,591,PRJNA768795,True
1,ICC2,360,prostate,35,449,PRJNA768795,True
2,ICC3,373,prostate,48,517,PRJNA768795,True
3,ICC4,521,prostate,25,655,PRJNA768795,True
4,ICC5,1106,prostate,94,1280,PRJNA768795,True
...,...,...,...,...,...,...,...
244,vdjNKI142,347,TNBC,0,347,bellini,True
245,vdjNKI146,1054,TNBC,130,1381,bellini,True
246,vdjNKI147,1477,TNBC,145,1782,bellini,True
247,vdjNKI148,1000,TNBC,95,1201,bellini,True


In [19]:
rename_dict = {
    "TNBC": "Breast",
    "BC": "Breast",
    "CRC": "Colon",
    "MEL": "Melanoma",
    "OV": "Ovary",
    "RCC": "Kidney clear cell",
    "pancreatic": "Pancreas",
    "PACA": "Pancreas",
    "BLA": "Bladder",
    "ESSC": "Esophagus",
    "THCA": "Thyroid",
    "UCEC": "Uterine",
    "ESCA": "Esophagus",
    "RC": "Kidney clear cell",
    "Lung": "NSCLC",
    "Endo": "Endometrium",
    "FTC": "Thyroid",
    'Renal': "Kidney clear cell",
    'breast': "Breast",
    'prostate': "Prostate"
    }

In [20]:
cd4_df["tumor_type"] = cd4_df["tumor_type"].replace(rename_dict)
cd4_df

,patient,n_cd4,tumor_type,more_or_equal_to_2,total_cd4_cells,data_source,CD4
0,ICC1,424,Prostate,47,591,PRJNA768795,True
1,ICC2,360,Prostate,35,449,PRJNA768795,True
2,ICC3,373,Prostate,48,517,PRJNA768795,True
3,ICC4,521,Prostate,25,655,PRJNA768795,True
4,ICC5,1106,Prostate,94,1280,PRJNA768795,True
...,...,...,...,...,...,...,...
244,vdjNKI142,347,Breast,0,347,bellini,True
245,vdjNKI146,1054,Breast,130,1381,bellini,True
246,vdjNKI147,1477,Breast,145,1782,bellini,True
247,vdjNKI148,1000,Breast,95,1201,bellini,True


In [ ]:
tumor_name_color_df = pd.read_csv('/PanCanAtlasTumors_color_coded_by_organ_system_20170302.tsv', sep='\t')

tumor_name_color_df.ton_name.dropna()

tumor_name_color_df

,Hex Colors,Study Abbreviation,Study Name,ton_name
0,#ED2891,BRCA,Breast invasive carcinoma,Breast
1,#B2509E,GBM,Glioblastoma multiforme,Glioblastoma
2,#D49DC7,LGG,Brain Lower Grade Glioma,NaN
3,#C1A72F,ACC,Adrenocortical carcinoma,NaN
4,#E8C51D,PCPG,Pheochromocytoma and Paraganglioma,NaN
5,#F9ED32,THCA,Thyroid carcinoma,NaN
6,#104A7F,CHOL,Cholangiocarcinoma,Cholangio
7,#9EDDF9,COAD,Colon adenocarcinoma,Colon
8,#007EB5,ESCA,Esophageal carcinoma,Esophagus
9,#CACCDB,LIHC,Liver hepatocellular carcinoma,Liver


In [22]:
unique_tumor_types = pd.Series(cd4_df.tumor_type.unique())
unique_tumor_types = unique_tumor_types[unique_tumor_types.isin(tumor_name_color_df.ton_name)]

tumor_name_color_df.loc[tumor_name_color_df.ton_name.isin(unique_tumor_types), :].loc[:, ['ton_name', 'Hex Colors']]

tumor_color_dict = dict(tumor_name_color_df.loc[tumor_name_color_df.ton_name.isin(unique_tumor_types), :].loc[:, ['ton_name', 'Hex Colors']].values.tolist()
)

tumor_color_dict.update({'Thyroid': '#F9ED32', 'NSCLC': '#A084BD', 'Prostate': '#7E1918'})
tumor_color_dict.update({'Lung adenocarcinoma': '#D3C3E0', 'Lung squamous': '#A084BD'})

tumor_color_dict


{'Breast': '#ED2891',
 'Colon': '#9EDDF9',
 'Esophagus': '#007EB5',
 'Pancreas': '#6E7BA2',
 'Endometrium': '#F6B667',
 'Ovary': '#D97D25',
 'Uterine': '#FBE3C7',
 'Melanoma': '#BBD642',
 'Bladder': '#FAD2D9',
 'Kidney clear cell': '#F8AFB3',
 'Thyroid': '#F9ED32',
 'NSCLC': '#A084BD',
 'Prostate': '#7E1918',
 'Lung adenocarcinoma': '#D3C3E0',
 'Lung squamous': '#A084BD'}

In [ ]:
bellini_df = pd.read_csv('/tcr_seq_data/NKI_datasets/bellini/outputs/bellini_unique_combined_meta_df.csv')
patient_list = list(bellini_df[~bellini_df.loc[:, 'Cohort_a'].isin(['C', 'D'])].patient.unique())

# remove non-relevant patients from bellini dataset
# now randomly downsample the marleen kok data

cd4_df.data_source.value_counts()

# biggest individual data  source is 18 

tmp_df = cd4_df[cd4_df.loc[:, 'data_source'] == 'bellini']
print(tmp_df.shape)

tmp_df = tmp_df[tmp_df.patient.isin(patient_list)]

print(tmp_df.shape)

df_downsample  = tmp_df.apply(lambda df: df.sample(n=18, random_state=1))

print(df_downsample.shape)
print(cd4_df.shape)
print(df_downsample.shape)
print(cd4_df[cd4_df.loc[:, 'data_source'] == 'bellini'].shape)
print(cd4_df[~(cd4_df.loc[:, 'data_source'] == 'bellini')].shape)

downsampled_filtered_cd4_df = pd.concat([df_downsample, cd4_df[~(cd4_df.loc[:, 'data_source'] == 'bellini')]])
print(downsampled_filtered_cd4_df.shape)

(71, 7)
(32, 7)
(18, 7)
(249, 7)
(18, 7)
(71, 7)
(178, 7)
(196, 7)


In [26]:
### check Uterine / endometrium names

downsampled_filtered_cd4_df.loc[:, 'tumor_type'].value_counts()

tmp_rename_dict = {
    'Uterine': 'Endometrium',
}

downsampled_filtered_cd4_df.loc[:, 'tumor_type'] = (
    downsampled_filtered_cd4_df['tumor_type'].replace(tmp_rename_dict))

In [ ]:
downsampled_filtered_cd4_df.loc[
    downsampled_filtered_cd4_df['patient'] == "OV",
    'tumor_type'] = 'Ovary'

downsampled_filtered_cd4_df.reset_index(drop=True, inplace=True)
downsampled_filtered_cd4_df.tumor_type.value_counts()

tumor_type
Breast               43
NSCLC                30
Kidney clear cell    20
Colon                18
Esophagus            14
Pancreas             12
Endometrium          12
Melanoma             11
Ovary                10
Thyroid              10
Bladder               9
Prostate              7
Name: count, dtype: int64

In [ ]:
adeno_list = ['Lung24', 'Lung26', 'Lung18', 'Lung15', 'Lung10', 'Lung11', 'Lung14', 'Lung16', 'Lung12', 'Lung17', 'Lung19', 'Lung1', 'Lung4', 'Lung5', 'Lung6','Lung7', 'Lung8']

squamous_list = ['Lung21', 'Lung22', 'Lung23', 'Lung25', 'Lung2', 'Lung3', 'Lung9']
mask = downsampled_filtered_cd4_df.patient.isin(adeno_list)

downsampled_filtered_cd4_df.loc[mask, "tumor_type"] = "Lung adenocarcinoma"
mask = downsampled_filtered_cd4_df.patient.isin(squamous_list)

downsampled_filtered_cd4_df.loc[mask, "tumor_type"] = "Lung squamous"
downsampled_filtered_cd4_df[downsampled_filtered_cd4_df.loc[:, 'tumor_type'] == "NSCLC"]
downsampled_filtered_cd4_df = downsampled_filtered_cd4_df[~downsampled_filtered_cd4_df.patient.isin(["Lung13", "Lung20", "Lung6"])]

In [29]:
downsampled_filtered_cd4_df.to_csv(os.path.join(output_dir, "cd4_patient_summary_downsampled.csv"), index=False)